# Laboratorio 5: Modelos de lenguaje

En este laboratorio se construyen modelos de lenguaje basados en n-gramas usando el texto de *Don Quijote de la Mancha*.  
El objetivo es preparar el corpus, construir modelos unigrama, bigrama y trigrama, aplicar suavizado, evaluar con perplejidad y probar una función simple de autocompletado.

## Importación de librerías

Primero se importan las librerías necesarias para leer el texto, limpiar el corpus, contar palabras y separar los datos en entrenamiento, validación y prueba.

In [3]:
# Librerías principales para manejo de texto, conteos y división de datos
import re
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

## Carga del corpus

Se carga el texto completo de *Don Quijote de la Mancha*.  
Antes de procesarlo, se muestra una pequeña parte para verificar que el archivo se leyó correctamente.

In [1]:
# Cargamos el texto completo de Don Quijote
with open("don-quijote.txt", "r", encoding="utf-8") as file:
    texto = file.read()

# Mostramos una parte pequeña del texto para verificar que se cargó bien
print(texto[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: Don Quijote

Author: Miguel de Cervantes Saavedra

Posting Date: April 27, 2010 [EBook #2000]
Release Date: December, 1999

Language: Spanish


*** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE ***




Produced by an anonymous Project Gutenberg volunteer. Text
file corrections and new HTML file by Joaquin Cuenca Abela.











El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada


## Limpieza básica

En esta parte solo se normalizan espacios y saltos de línea.  
No se eliminan stopwords ni se aplica lematización, porque el objetivo es conservar el orden natural de las palabras para construir modelos de lenguaje.

In [4]:
# Reemplazamos saltos de línea múltiples por espacios
texto_limpio = re.sub(r"\s+", " ", texto)

# Quitamos espacios al inicio y al final
texto_limpio = texto_limpio.strip()

# Verificamos el resultado
print(texto_limpio[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer. Text file corrections and new HTML file by Joaquin Cuenca Abela. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho libro a t


## Segmentación en oraciones

El texto se divide en oraciones porque los modelos de lenguaje trabajan con secuencias.  
Cada oración será tratada como una secuencia independiente de palabras.

In [6]:
# Dividimos el texto en oraciones usando puntos, signos de pregunta y exclamación
oraciones = re.split(r'(?<=[.!?¿¡])\s+', texto_limpio)

# Eliminamos oraciones vacías
oraciones = [oracion.strip() for oracion in oraciones if oracion.strip()]

# Mostramos cuántas oraciones se obtuvieron
print("Cantidad de oraciones:", len(oraciones))

# Ejemplo de algunas oraciones
for i in range(5):
    print(f"{i+1}.", oraciones[i])

Cantidad de oraciones: 9577
1. ﻿The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.
2. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer.
3. Text file corrections and new HTML file by Joaquin Cuenca Abela.
4. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra,